### ЗАДАЧА: Диспетчер фулфилмента

Есть поток заказов на сборку и набор команд смены.
Нужно собрать диспетчер, который:
- держит срочные и обычные заказы в отдельных очередях,
- выдаёт заказы в работу по приоритету,
- упаковывает заказ и ведёт статистику,
- умеет откатывать последнее успешное действие через стек,
- обновляет дерево категорий по каждому упакованному заказу.

In [ ]:
from collections import deque
from copy import deepcopy

# row format: order_id|client|priority|category_path|items
rows = [
    "ORD-900|Alice|express|electronics/phones|2",
    "ORD-901|Bob|normal|home/kitchen|4",
    "ORD-902|Dina|express|electronics/laptops|1",
    "ORD-903|Max|normal|home/decor|3",
    "ORD-904|Olga|normal|sports/running|2",
    "ORD-905|Ira|express|sports/cycling|1",
]

# command format:
# - pull_next
# - pack|operator|minutes
# - undo
# - report
commands = [
    "pull_next",
    "pack|Nikita|12",
    "pull_next",
    "pack|Olga|9",
    "pull_next",
    "report",
    "pack|Nikita|8",
    "undo",
    "pull_next",
    "pack|Ira|11",
    "pull_next",
    "pack|Olga|7",
    "report",
]

category_tree = {
    "name": "root",
    "count": 0,
    "children": {
        "electronics": {
            "name": "electronics",
            "count": 0,
            "children": {
                "phones": {"name": "phones", "count": 0, "children": {}},
                "laptops": {"name": "laptops", "count": 0, "children": {}},
            },
        },
        "home": {
            "name": "home",
            "count": 0,
            "children": {
                "kitchen": {"name": "kitchen", "count": 0, "children": {}},
                "decor": {"name": "decor", "count": 0, "children": {}},
            },
        },
        "sports": {
            "name": "sports",
            "count": 0,
            "children": {
                "running": {"name": "running", "count": 0, "children": {}},
                "cycling": {"name": "cycling", "count": 0, "children": {}},
            },
        },
    },
}

orders = []
express_queue = deque()
normal_queue = deque()

current_order = None
packed_orders = []
operator_minutes = {}
history_stack = []
reports = []
errors = []
undo_count = 0

def build_order(row):
    order_id, client, priority, category_path_raw, items_raw = row.split("|")
    return {
        "order_id": order_id,
        "client": client,
        "priority": priority,
        "category_path": category_path_raw.split("/"),
        "items": int(items_raw),
    }

def push_snapshot():
    snapshot = {
        "current_order": deepcopy(current_order),
        "packed_orders": deepcopy(packed_orders),
        "operator_minutes": deepcopy(operator_minutes),
        "category_tree": deepcopy(category_tree),
    }
    history_stack.append(snapshot)

def restore_snapshot(snapshot):
    global current_order, packed_orders, operator_minutes, category_tree
    current_order = snapshot["current_order"]
    packed_orders = snapshot["packed_orders"]
    operator_minutes = snapshot["operator_minutes"]
    category_tree = snapshot["category_tree"]

def update_category_counts(path_parts):
    # TODO 1: увеличьте node['count'] у корня перед проходом по path_parts
    node = category_tree
    node['count'] += 1

    # TODO 2: для каждого part из path_parts:
    for part in path_parts:
        # - проверьте, что part есть в node['children']
        if part not in node['children']:
            # - если нет, верните False
            return False
        # - если есть, перейдите к дочернему узлу
        node = node['children'][part]
        # - увеличьте у него поле 'count' на 1
        node['count'] += 1
    # TODO 3: если весь путь успешно пройден, верните True
    return True

def pull_next_order():
    # TODO 4: если express_queue не пуст, вернуть express_queue.popleft()
    if express_queue:
        return express_queue.popleft()
    # TODO 5: иначе если normal_queue не пуст, вернуть normal_queue.popleft()
    elif normal_queue:
        return normal_queue.popleft()
    # TODO 6: если обе очереди пусты, вернуть None
    else:
        return None

for row in rows:
    order = build_order(row)
    orders.append(order)

    # TODO 7: разложите orders по двум очередям:
    if order['priority'] == 'express':
        express_queue.append(order)  # - priority == 'express' -> express_queue.append(order)
    else:
        normal_queue.append(order)  # - иначе -> normal_queue.append(order)

for command in commands:
    if command == "pull_next":
        # TODO 8: если current_order уже есть,
        if current_order is not None:
            # добавьте ошибку о том, что заказ уже взят в работу, и continue
            errors.append("Заказ уже взят в работу")
            continue
        # TODO 9: получите следующий заказ через pull_next_order()
        next_order = pull_next_order()
        # TODO 10: если заказ найден, положите его в current_order
        if next_order:
            current_order = next_order
        # TODO 11: если заказа нет, добавьте ошибку "Очереди пусты"
        else:
            errors.append("Очереди пусты")
        continue

    if command == "undo":
        # TODO 12: если history_stack не пуст,
        if history_stack:
            # достаньте последний snapshot через pop,
            snapshot = history_stack.pop()
            # восстановите состояние через restore_snapshot,
            restore_snapshot(snapshot)
            # увеличьте undo_count
            undo_count += 1
        # TODO 13: если стек пуст, добавьте ошибку "Нечего откатывать"
        else:
            errors.append("Нечего откатывать")
        continue

    if command == "report":
        # TODO 14: найдите самого загруженного оператора:
        top_operator = None
        top_minutes = 0
        for operator, minutes in operator_minutes.items():
            if minutes > top_minutes:
                top_minutes = minutes
                top_operator = operator
        # TODO 15: добавьте в reports словарь со срезом состояния:
        reports.append({
            'current_order': current_order['order_id'] if current_order else None,
            'express_left': len(express_queue),
            'normal_left': len(normal_queue),
            'packed_total': len(packed_orders),
            'top_operator': top_operator,
            'top_minutes': top_minutes,
        })
        continue

    if command.startswith("pack|"):
        _, operator, minutes_raw = command.split("|")
        minutes = int(minutes_raw)

        # TODO 16: если current_order is None,
        if current_order is None:
            # добавьте ошибку "Нет заказа в работе" и continue
            errors.append("Нет заказа в работе")
            continue
        # TODO 17: перед любыми изменениями вызовите push_snapshot()
        push_snapshot()
        # TODO 18: обновите operator_minutes для operator,
        operator_minutes[operator] = operator_minutes.get(operator, 0) + minutes
        # TODO 19: добавьте запись в packed_orders:
        packed_orders.append({
            'order_id': current_order['order_id'],
            'client': current_order['client'],
            'operator': operator,
            'minutes': minutes,
            'items': current_order['items'],
            'category_path': '/'.join(current_order['category_path']),
        })
        # TODO 20: обновите category_tree через update_category_counts(current_order['category_path'])
        success = update_category_counts(current_order['category_path'])
        # TODO 21: если update_category_counts вернул False,
        if not success:
            # добавьте ошибку с order_id и сделайте откат через restore_snapshot(history_stack.pop())
            errors.append(f"Ошибка обновления категорий для заказа {current_order['order_id']}")
            restore_snapshot(history_stack.pop())
        else:
            # TODO 22: если обновление успешно, очистите current_order = None
            current_order = None
        continue

    errors.append(f"Неизвестная команда: {command}")

print("Итог упакованных заказов:", packed_orders)
print("Минуты по операторам:", operator_minutes)
print("Отчёты смены:", reports)
print("Ошибки:", errors)
print("Успешных undo:", undo_count)
print("Категории (root count):", category_tree["count"])
